# 7. Machine learning: generating and completing spectra

Two small neural networks (PyTorch; `pip install "sed-tools[full]"`):
- the **generator** makes a whole spectrum from Teff, log g and [M/H] alone;
- the **completer** extends a spectrum with limited wavelength coverage to a wider range.

Both train here on one variant of the Kurucz cut (425 spectra) for at most 200 epochs, which takes seconds. Small training gives rough results; the point is the workflow. Train on a larger grid for real use.

Models are saved under `<stellar models>/models/<name>/`.

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np

from sed_tools.api import SED

plt.rcParams["figure.dpi"] = 80
SIGMA_SB = 5.670374419e-5
KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)
# Notebook 1 downloads these grids; they are fetched here only if missing, so this notebook needs no network.
for name, cut in (("Kurucz2003all", KURUCZ_CUT),):
    if not (SED._model_root / name / "lookup_table.csv").is_file():
        SED.fetch(name, source="njm", **cut)
TRAIN = "Kurucz2003all/Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s"
OTHER = "Kurucz2003all/Kurucz2003all_alpha_0p40_lh_1p25_vtur_2_km_s"
reference = SED.local(TRAIN)

## 1. The generator

`train(grid=...)` reads the grid's flux cube. The parameter ranges it was trained on are stored with the model; spectra the training could not use (negative or non-finite flux, or less than 30 % of its wavelength grid covered) are dropped and counted in the output; zeros are kept and left out of the fit.

In [ ]:
import random

import torch

# A fixed seed (the sample draw, the batch order and the starting weights), so every run trains the same
# model. Seeds 0-5 gave a bolometric ratio below of 0.86-2.03; 200 epochs is a short training.
SEED = 4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
generator = SED.ml_generator()
generator.train(grid=TRAIN, save_name="nb_generator", epochs=200)
print(generator.parameter_ranges())

In [ ]:
generator = SED.ml_generator().load("nb_generator")
wavelength, flux = generator.generate(teff=5777, logg=4.44, metallicity=0.0)
truth = reference(5777, 4.44, 0.0)
truth_flux = np.interp(wavelength, truth.wavelength, truth.flux)
optical = (wavelength > 3000) & (wavelength < 20000)
print(f"median |generated / grid − 1| at 3000–20000 Å: {np.median(np.abs(flux[optical] / truth_flux[optical] - 1)):.2f}")
ratio = np.trapezoid(flux, wavelength) / np.trapezoid(truth_flux, wavelength)
print(f"bolometric flux, generated / grid: {ratio:.3f}")
assert np.all(np.isfinite(flux)) and 0.5 < ratio < 2

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.loglog(truth.wavelength, truth.flux, lw=0.7, label="grid (interpolated)")
ax.loglog(wavelength, flux, lw=0.9, label="generator")
ax.set(xlim=(1000, 1e5), ylim=(1e3, 2e7), xlabel="Wavelength (Å)", ylabel="F$_\\lambda$", title="5777 K, log g 4.44")
ax.legend();

`generate_with_outputs` also writes the spectrum and two diagnostic plots. A generator can fill new nodes when densifying a cube (`densify_grid(..., ml_model=<model folder>)`, notebook 5).

In [ ]:
out_dir = SED._model_root / "models" / "nb_generator" / "SED"
generator.generate_with_outputs(teff=6000, logg=4.0, metallicity=-0.5, output_dir=str(out_dir))
print(sorted(p.name for p in out_dir.iterdir()))

## 2. The completer

The completer reads a central window of its wavelength grid (about 800–12600 Å with the defaults) and predicts the rest. A spectrum to extend should cover at least 80 % of that window, as the training spectra did; with less, it warns, and the result is not reliable.

To have a known answer, cut the alpha = 0.4 spectra at log g 4.5, [M/H] 0 to 700–13000 Å, bring them in as a grid of their own with `SED.import_grid`, and extend them back to 100–100000 Å.

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)   # the input windows, batch order and weights
completer = SED.ml_completer()
completer.train(grid=TRAIN, save_name="nb_completer", epochs=200)

In [ ]:
cut_dir = SED._model_root.parent / "cut_spectra"
cut_dir.mkdir(exist_ok=True)
full = [s for s in SED.local(OTHER).grid if s.logg == 4.5 and s.metallicity == 0.0]
for s in full:
    keep = (s.wavelength >= 700) & (s.wavelength <= 13000)
    dataclasses.replace(s, wavelength=s.wavelength[keep], flux=s.flux[keep]).save(cut_dir / s.filename)

result = SED.import_grid(cut_dir, name="Kurucz_cut", flux_unit="flam")
print(result)

In [ ]:
completer = SED.ml_completer().load("nb_completer")
extended = completer.extend("Kurucz_cut", wavelength_range=(100, 100000))
print(len(extended), "spectra extended")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
by_teff = {s.teff: s for s in full}
for s in extended:
    if s.teff not in (4500, 6000, 7500):
        continue
    original = by_teff[s.teff]
    ax.loglog(original.wavelength, original.flux, color="0.6", lw=0.6)
    ax.loglog(s.wavelength, s.flux, lw=0.9, label=f"{s.teff:.0f} K extended")
    infrared = (s.wavelength > 15000) & (s.wavelength < 90000)
    known = np.interp(s.wavelength[infrared], original.wavelength, original.flux)
    error = np.median(np.abs(s.flux[infrared] / known - 1))
    print(f"{s.teff:.0f} K: median |extended / original − 1| at 1.5–9 µm: {error:.2f}")
    # The seeded model gives 0.17-0.23 here; a median error above 35 % means the completer is not working.
    assert np.all(np.isfinite(s.flux)) and error < 0.35
ax.axvspan(700, 13000, color="0.9", zorder=0)
ax.set(xlim=(100, 1e5), ylim=(1e-2, 1e8), xlabel="Wavelength (Å)", ylabel="F$_\\lambda$",
       title="grey: the full spectra; shaded: what the completer was given")
ax.legend(fontsize=8);

## Command-line equivalents

| Python | Command line |
|---|---|
| `SED.ml_generator().train(...)`, `.generate(...)` | `sed-tools ml_generator` (interactive) |
| hyperparameter search, then training | `sed-tools ml_generator --auto --library GRID_FOLDER --output OUT --trials 20 --epochs 100` (needs `sed-tools[full,ml-tune]`) |
| `SED.ml_completer().train(...)`, `.extend(...)` | `sed-tools ml_completer` (interactive) |
| `SED.import_grid(folder, name=N, flux_unit="flam")` | `sed-tools import --path FOLDER --name N --flux-unit flam` |